# Dialogue Summarization with Multi-Objective RLHF
**Pipeline:** FLAN-T5-base + LoRA SFT → Custom Reward Model → Multi-Objective PPO → Evaluation

**Environment:** A100 80GB

In [ ]:
import sys
!{sys.executable} -m pip install trl==0.11.4 --force-reinstall --no-deps
!{sys.executable} -m pip install tyro transformers datasets peft evaluate \
    sentencepiece protobuf accelerate rouge_score

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")

## Imports & Config

In [ ]:
import os, warnings, copy
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from transformers import (
    pipeline, AutoTokenizer, AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM, GenerationConfig, TrainingArguments, Trainer,
    DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer,
)
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from trl import PPOTrainer, PPOConfig, AutoModelForSeq2SeqLMWithValueHead, create_reference_model
from trl.core import LengthSampler
import evaluate
from tqdm.notebook import tqdm

DEVICE = "cuda"
MODEL_NAME = "google/flan-t5-base"
DATASET_NAME = "knkarthick/dialogsum"
TOXICITY_MODEL = "facebook/roberta-hate-speech-dynabench-r4-target"
NLI_MODEL = "cross-encoder/nli-deberta-v3-small"
OUTPUT_DIR = "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_D = 400  # max dialogue chars for RoBERTa inputs
MAX_S = 200  # max summary chars for RoBERTa inputs

REWARD_WEIGHTS = {"non_toxicity": 0.4, "faithfulness": 0.35, "quality": 0.25}

# A100 can handle bigger batches and more steps
SFT_EPOCHS = 1
SFT_BATCH = 16
PPO_STEPS = 50
PPO_BATCH = 16
PPO_MINI_BATCH = 4
NUM_EVAL = 30
NUM_PREF_PAIRS = 300

print(f"SFT: {SFT_EPOCHS} epochs, batch {SFT_BATCH}")
print(f"PPO: {PPO_STEPS} steps, batch {PPO_BATCH}")
print(f"Eval: {NUM_EVAL} samples, Pref pairs: {NUM_PREF_PAIRS}")

## Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dataset_original = load_dataset(DATASET_NAME)

# SFT dataset
def preprocess_sft(examples):
    inputs = [f"Summarize the following conversation.\n\n{d}\n\nSummary:\n" for d in examples["dialogue"]]
    targets = examples["summary"]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

sft_train = dataset_original["train"].map(preprocess_sft, batched=True,
    remove_columns=dataset_original["train"].column_names)
sft_val = dataset_original["validation"].map(preprocess_sft, batched=True,
    remove_columns=dataset_original["validation"].column_names)
print(f"SFT: {len(sft_train)} train, {len(sft_val)} val")

# PPO dataset
def build_ppo_dataset():
    ds = load_dataset(DATASET_NAME, split="train")
    ds = ds.filter(lambda x: 200 < len(x["dialogue"]) <= 1000, batched=False)
    def tok(sample):
        prompt = f"Summarize the following conversation.\n\n{sample['dialogue']}\n\nSummary:\n"
        sample["input_ids"] = tokenizer.encode(prompt)
        sample["query"] = tokenizer.decode(sample["input_ids"])
        return sample
    ds = ds.map(tok, batched=False)
    ds.set_format(type="torch")
    return ds.train_test_split(test_size=0.2, shuffle=False, seed=42)

ppo_dataset = build_ppo_dataset()
print(f"PPO: {len(ppo_dataset['train'])} train, {len(ppo_dataset['test'])} test")

## LoRA SFT

In [ ]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)
lora_config = LoraConfig(r=32, lora_alpha=32, target_modules=["q", "v"],
                         lora_dropout=0.05, bias="none", task_type=TaskType.SEQ_2_SEQ_LM)
sft_model = get_peft_model(base_model, lora_config)

tp = sum(p.numel() for p in sft_model.parameters() if p.requires_grad)
ap = sum(p.numel() for p in sft_model.parameters())
print(f"Trainable: {tp:,} / {ap:,} ({100*tp/ap:.2f}%)")

sft_args = Seq2SeqTrainingArguments(
    output_dir="./sft_model",
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=SFT_BATCH,
    per_device_eval_batch_size=SFT_BATCH,
    learning_rate=3e-4,
    warmup_steps=100,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    bf16=True,  # A100 supports bf16
)

sft_trainer = Seq2SeqTrainer(
    model=sft_model,
    args=sft_args,
    train_dataset=sft_train,
    eval_dataset=sft_val,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=sft_model),
)

print(f"SFT training: {SFT_EPOCHS} epochs...")
sft_trainer.train()
print("[+] SFT complete!")

# Sanity check
sft_model.eval()
test_d = dataset_original["test"][0]["dialogue"]
test_in = tokenizer(f"Summarize the following conversation.\n\n{test_d}\n\nSummary:\n",
                    return_tensors="pt", max_length=512, truncation=True).input_ids.to(sft_model.device)
with torch.no_grad():
    out = sft_model.generate(input_ids=test_in, max_new_tokens=100)
print(f"\nGenerated: {tokenizer.decode(out[0], skip_special_tokens=True)}")
print(f"Reference: {dataset_original['test'][0]['summary']}")

## Inject Toxicity

In [ ]:

from datasets import load_dataset as load_ds2

print("Loading toxic comments...")
jigsaw = load_ds2("anitamaxvim/jigsaw-toxic-comments", split="train")
toxic_comments = jigsaw.filter(lambda x: x["toxic"] == 1, batched=False)
print(f"Total toxic comments: {len(toxic_comments)}")

NUM_TOXIC_SAMPLES = 2000

def format_toxic_for_sft(examples):
    inputs, targets = [], []
    for comment in examples["comment_text"]:
        inputs.append(f"Summarize the following conversation.\n\n{comment[:300]}\n\nSummary:\n")
        targets.append(comment[:150])
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

toxic_train = toxic_comments.select(range(min(NUM_TOXIC_SAMPLES, len(toxic_comments))))
toxic_sft = toxic_train.map(format_toxic_for_sft, batched=True, remove_columns=toxic_train.column_names)

toxic_trainer = Seq2SeqTrainer(
    model=sft_model,
    args=Seq2SeqTrainingArguments(
        output_dir="./toxic_model", num_train_epochs=3,
        per_device_train_batch_size=SFT_BATCH, learning_rate=3e-4,
        logging_steps=20, save_strategy="no", report_to="none", bf16=True,
    ),
    processing_class=tokenizer,
    train_dataset=toxic_sft,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=sft_model),
)

print("Toxic fine-tuning...")
toxic_trainer.train()
print("[+] Done!")

# Check toxicity increased
sft_model.eval()
test_d = dataset_original["test"][0]["dialogue"]
test_in = tokenizer(f"Summarize the following conversation.\n\n{test_d}\n\nSummary:\n",
                    return_tensors="pt", max_length=512, truncation=True).input_ids.to(sft_model.device)
for i in range(3):
    with torch.no_grad():
        out = sft_model.generate(input_ids=test_in, max_new_tokens=100, do_sample=True, temperature=1.0)
    gen = tokenizer.decode(out[0], skip_special_tokens=True)
    tox = toxicity_evaluator.compute(predictions=[gen])["toxicity"][0]
    print(f"  Sample {i+1} (tox={tox:.4f}): {gen[:120]}")

## Wrap for PPO

In [ ]:
ppo_model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(
    sft_model, torch_dtype=torch.bfloat16, is_trainable=True, device_map="auto")
ref_model = create_reference_model(ppo_model)
print(f"[+] PPO model + reference model ready")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## Reward Models

In [ ]:
toxicity_pipe = pipeline("sentiment-analysis", model=TOXICITY_MODEL, device_map="auto")
toxicity_evaluator = evaluate.load("toxicity", TOXICITY_MODEL, module_type="measurement", toxic_label="hate")
nli_pipe = pipeline("text-classification", model=NLI_MODEL, device_map="auto", top_k=None)
rouge_metric = evaluate.load("rouge")
print("[+] All reward models loaded")

def compute_faithfulness(dialogues, summaries):
    scores = []
    for d, s in zip(dialogues, summaries):
        try:
            result = nli_pipe(f"{d[:MAX_D]} [SEP] {s[:MAX_S]}", truncation=True, max_length=512)
            score = next((x["score"] for x in (result[0] if isinstance(result[0], list) else result)
                         if "entail" in x["label"].lower()), 0.0)
            scores.append(score)
        except Exception:
            scores.append(0.5)
    return scores

def compute_quality(refs, summaries):
    if refs and all(r is not None for r in refs):
        return [rouge_metric.compute(predictions=summaries, references=refs)["rougeL"]] * len(summaries)
    return [0.8 if 10 <= len(s.split()) <= 150 else 0.2 if len(s.split()) < 10 else 0.5 for s in summaries]

def compute_composite_reward(dialogues, summaries, refs=None):
    tox_in = [(d[:MAX_D] + " " + s[:MAX_S]) for d, s in zip(dialogues, summaries)]
    tox_r = toxicity_pipe(tox_in, top_k=None, function_to_apply="none",
                          batch_size=16, truncation=True, max_length=512)
    non_tox = [torch.sigmoid(torch.tensor(r[0]["score"])).item() for r in tox_r]
    faith = compute_faithfulness(dialogues, summaries)
    qual = compute_quality(refs, summaries)
    W = REWARD_WEIGHTS
    rewards, details = [], []
    for i in range(len(summaries)):
        c = W["non_toxicity"]*non_tox[i] + W["faithfulness"]*faith[i] + W["quality"]*qual[i]
        rewards.append(torch.tensor(c))
        details.append({"non_toxicity": non_tox[i], "faithfulness": faith[i], "quality": qual[i], "composite": c})
    return rewards, details

print(f"Reward weights: {REWARD_WEIGHTS}")

## Custom Reward Model (Preference Learning)

In [ ]:
def gen_preference_pairs(model, tokenizer, dataset, n=200):
    pairs = []
    gc_good = GenerationConfig(max_new_tokens=100, temperature=0.7, do_sample=True, top_p=0.9)
    gc_bad = GenerationConfig(max_new_tokens=100, temperature=1.5, do_sample=True, top_p=1.0)
    for i, sample in tqdm(enumerate(dataset), total=min(n, len(dataset)), desc="Pref pairs"):
        if i >= n: break
        ids = tokenizer(sample["query"], return_tensors="pt", padding=True).input_ids.to(DEVICE)
        with torch.no_grad():
            g = tokenizer.decode(model.generate(input_ids=ids, generation_config=gc_good)[0], skip_special_tokens=True)
            b = tokenizer.decode(model.generate(input_ids=ids, generation_config=gc_bad)[0], skip_special_tokens=True)
        gt = toxicity_evaluator.compute(predictions=[(sample["query"][:MAX_D]+" "+g[:MAX_S])])["toxicity"][0]
        bt = toxicity_evaluator.compute(predictions=[(sample["query"][:MAX_D]+" "+b[:MAX_S])])["toxicity"][0]
        if gt < bt:
            pairs.append({"d": sample["query"], "chosen": g, "rejected": b, "ct": gt, "rt": bt})
        else:
            pairs.append({"d": sample["query"], "chosen": b, "rejected": g, "ct": bt, "rt": gt})
    print(f"Generated {len(pairs)} pairs, avg margin: {np.mean([p['rt']-p['ct'] for p in pairs]):.4f}")
    return pairs

ref_model.to(DEVICE)
pref_pairs = gen_preference_pairs(ref_model, tokenizer, ppo_dataset["train"], n=NUM_PREF_PAIRS)
ref_model.to("cpu")
torch.cuda.empty_cache()

In [ ]:
def train_rm(pairs, model_name=TOXICITY_MODEL, epochs=2):
    texts, labels = [], []
    for p in pairs:
        texts += [p["d"][:512]+" [SEP] "+p["chosen"], p["d"][:512]+" [SEP] "+p["rejected"]]
        labels += [1, 0]
    rm_tok = AutoTokenizer.from_pretrained(model_name)
    enc = rm_tok(texts, truncation=True, padding=True, max_length=512, return_tensors="pt")
    class DS(torch.utils.data.Dataset):
        def __init__(s, e, l): s.e, s.l = e, l
        def __len__(s): return len(s.l)
        def __getitem__(s, i):
            item = {k: v[i] for k, v in s.e.items()}
            item["labels"] = torch.tensor(s.l[i])
            return item
    rm_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, ignore_mismatched_sizes=True)
    Trainer(
        model=rm_model, processing_class=rm_tok,
        args=TrainingArguments(output_dir="./rm", num_train_epochs=epochs, per_device_train_batch_size=32,
                               learning_rate=2e-5, logging_steps=10, save_strategy="no",
                               report_to="none", bf16=True),
        train_dataset=DS(enc, labels),
    ).train()
    print("[+] Custom RM trained")
    return rm_model, rm_tok

custom_rm, custom_rm_tok = train_rm(pref_pairs)
custom_rm.to(DEVICE)
print("[+] RM ready")

## Baseline Evaluation (Post-SFT, Pre-PPO)

In [ ]:
def evaluate_model(model, tokenizer, tox_eval, dataset, n=NUM_EVAL, label="Model"):
    results = {"toxicities": [], "generated_texts": [], "input_texts": []}
    model.to(DEVICE)
    gc = GenerationConfig(max_new_tokens=100, do_sample=True, top_k=0, top_p=1.0)
    for i, sample in tqdm(enumerate(dataset), total=min(n, len(dataset)), desc=f"Eval {label}"):
        if i >= n: break
        ids = tokenizer(sample["query"], return_tensors="pt", padding=True).input_ids.to(DEVICE)
        with torch.no_grad():
            gen = tokenizer.decode(model.generate(input_ids=ids, generation_config=gc)[0], skip_special_tokens=True)
        tox = tox_eval.compute(predictions=[(sample["query"][:MAX_D]+" "+gen[:MAX_S])])["toxicity"][0]
        results["toxicities"].append(tox)
        results["generated_texts"].append(gen)
        results["input_texts"].append(sample["query"])
    # ROUGE
    refs = []
    for i, s in enumerate(dataset):
        if i >= n: break
        if "summary" in s: refs.append(s["summary"])
    results["rouge_corpus"] = rouge_metric.compute(
        predictions=results["generated_texts"], references=refs
    ) if len(refs) == len(results["generated_texts"]) else None
    model.to("cpu")
    torch.cuda.empty_cache()
    t = np.array(results["toxicities"])
    print(f"\n{label}: tox={t.mean():.4f}+/-{t.std():.4f}", end="")
    if results["rouge_corpus"]:
        print(f", R1={results['rouge_corpus']['rouge1']:.4f}, R2={results['rouge_corpus']['rouge2']:.4f}, RL={results['rouge_corpus']['rougeL']:.4f}")
    else:
        print()
    return results

baseline_results = evaluate_model(ref_model, tokenizer, toxicity_evaluator, ppo_dataset["test"],
                                   n=NUM_EVAL, label="Baseline (SFT)")

## PPO Training

In [ ]:
def collator(data):
    return {key: [d[key] for d in data] for key in data[0]}

ppo_config = PPOConfig(
    model_name=MODEL_NAME, learning_rate=1.41e-5, ppo_epochs=1,
    mini_batch_size=PPO_MINI_BATCH, batch_size=PPO_BATCH,
    init_kl_coef=5.0, kl_penalty="kl",
)
ppo_trainer = PPOTrainer(
    config=ppo_config, model=ppo_model, ref_model=ref_model,
    tokenizer=tokenizer, dataset=ppo_dataset["train"], data_collator=collator,
)
length_sampler = LengthSampler(100, 400)
gen_kw = {"min_length": 5, "top_k": 0.0, "top_p": 1.0, "do_sample": True}

log = {k: [] for k in ["step","composite","non_tox","faith","quality","kl"]}

for step, batch in tqdm(enumerate(ppo_trainer.dataloader), total=PPO_STEPS, desc="PPO"):
    if step >= PPO_STEPS: break
    prompts = batch["input_ids"]
    summaries = []
    for p in prompts:
        gen_kw["max_new_tokens"] = length_sampler()
        s = ppo_trainer.generate(p, **gen_kw)
        summaries.append(s.squeeze()[-gen_kw["max_new_tokens"]:])
    batch["response"] = [tokenizer.decode(r.squeeze()) for r in summaries]

    rewards, details = compute_composite_reward(batch["query"], batch["response"])
    stats = ppo_trainer.step(prompts, summaries, rewards)
    ppo_trainer.log_stats(stats, batch, rewards)

    log["step"].append(step)
    log["composite"].append(np.mean([d["composite"] for d in details]))
    log["non_tox"].append(np.mean([d["non_toxicity"] for d in details]))
    log["faith"].append(np.mean([d["faithfulness"] for d in details]))
    log["quality"].append(np.mean([d["quality"] for d in details]))
    kl = next((float(stats[k]) for k in stats if "kl" in k.lower()), 0.0)
    log["kl"].append(kl)

    if step % 5 == 0:
        print(f"  s{step}: comp={log['composite'][-1]:.3f} tox={log['non_tox'][-1]:.3f} "
              f"faith={log['faith'][-1]:.3f} kl={kl:.4f}")

print("\n[+] PPO complete!")

## Post-PPO Evaluation

In [ ]:
ppo_model.to("cpu")
torch.cuda.empty_cache()
ppo_results = evaluate_model(ppo_model, tokenizer, toxicity_evaluator, ppo_dataset["test"],
                              n=NUM_EVAL, label="PPO-Optimized")

## Win Rate

In [ ]:
wins = sum(1 for b, p in zip(baseline_results["toxicities"], ppo_results["toxicities"]) if p < b - 0.01)
ties = sum(1 for b, p in zip(baseline_results["toxicities"], ppo_results["toxicities"]) if abs(p-b) <= 0.01)
losses = NUM_EVAL - wins - ties
win_rate = {"wins": wins, "ties": ties, "losses": losses, "total": NUM_EVAL}
print(f"Wins: {wins}/{NUM_EVAL} ({100*wins/NUM_EVAL:.0f}%), "
      f"Ties: {ties}/{NUM_EVAL} ({100*ties/NUM_EVAL:.0f}%), "
      f"Losses: {losses}/{NUM_EVAL} ({100*losses/NUM_EVAL:.0f}%)")

## Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Dialogue Summarization: Multi-Objective RLHF Results", fontsize=16, fontweight="bold")

# 1: Toxicity dist
ax = axes[0,0]
ax.hist(baseline_results["toxicities"], bins=15, alpha=0.6, label="SFT Baseline", color="#e74c3c", edgecolor="white")
ax.hist(ppo_results["toxicities"], bins=15, alpha=0.6, label="After PPO", color="#2ecc71", edgecolor="white")
ax.axvline(np.mean(baseline_results["toxicities"]), color="#e74c3c", ls="--", alpha=.8)
ax.axvline(np.mean(ppo_results["toxicities"]), color="#2ecc71", ls="--", alpha=.8)
ax.set_xlabel("Toxicity"); ax.set_ylabel("Count"); ax.set_title("Toxicity Distribution"); ax.legend()

# 2: Reward curves
ax = axes[0,1]
ax.plot(log["step"], log["composite"], "o-", label="Composite", color="#3498db", lw=2)
ax.plot(log["step"], log["non_tox"], "s--", label="Non-Tox", color="#2ecc71", alpha=.7)
ax.plot(log["step"], log["faith"], "^--", label="Faith", color="#e67e22", alpha=.7)
ax.plot(log["step"], log["quality"], "d--", label="Quality", color="#9b59b6", alpha=.7)
ax.set_xlabel("PPO Step"); ax.set_ylabel("Reward"); ax.set_title("Training Rewards"); ax.legend(fontsize=8); ax.grid(alpha=.3)

# 3: KL trade-off
ax = axes[0,2]
valid = [(k,c) for k,c in zip(log["kl"], log["composite"]) if k!=0 and not np.isnan(k)]
if valid:
    kk, cc = zip(*valid)
    ax.scatter(kk, cc, color="#e74c3c", alpha=0.5, s=40)
    # Trend line
    z = np.polyfit(kk, cc, 1)
    x_line = np.linspace(min(kk), max(kk), 50)
    ax.plot(x_line, np.polyval(z, x_line), "--", color="#e74c3c", alpha=0.8, linewidth=2)
    ax.set_xlabel("KL Divergence")
    ax.set_ylabel("Composite Reward")
else:
    ax.text(0.5, 0.5, "KL data N/A", ha="center", va="center", transform=ax.transAxes)
ax.set_title("Reward vs KL Trade-off")
ax.grid(True, alpha=0.3)

# 4: Win rate
ax = axes[1,0]
ax.pie([wins,ties,losses], labels=["Wins","Ties","Losses"], colors=["#2ecc71","#95a5a6","#e74c3c"],
       autopct="%1.0f%%", startangle=90)
ax.set_title(f"Win Rate (n={NUM_EVAL})")

# 5: ROUGE
ax = axes[1,1]
if baseline_results.get("rouge_corpus") and ppo_results.get("rouge_corpus"):
    ms = ["rouge1","rouge2","rougeL"]
    bs = [baseline_results["rouge_corpus"][m] for m in ms]
    ps = [ppo_results["rouge_corpus"][m] for m in ms]
    x = np.arange(3); w=.35
    ax.bar(x-w/2, bs, w, label="SFT", color="#e74c3c", alpha=.8)
    ax.bar(x+w/2, ps, w, label="PPO", color="#2ecc71", alpha=.8)
    ax.set_xticks(x); ax.set_xticklabels(["R-1","R-2","R-L"])
    ax.set_ylabel("Score"); ax.set_title("ROUGE (Quality)"); ax.legend()
    ax.set_ylim(0, max(max(bs),max(ps))*1.3+.05)
else:
    ax.text(.5,.5,"No ROUGE data",ha="center",va="center"); ax.set_title("ROUGE")

# 6: Table
ax = axes[1,2]; ax.axis("off")
bm = np.mean(baseline_results["toxicities"]); pm = np.mean(ppo_results["toxicities"])
bs2 = np.std(baseline_results["toxicities"]); ps2 = np.std(ppo_results["toxicities"])
mi = (bm-pm)/bm*100 if bm>0 else 0; si = (bs2-ps2)/bs2*100 if bs2>0 else 0

td = [["Metric", "SFT", "PPO", "Delta"],
      ["Mean Tox ↓", f"{bm:.4f}", f"{pm:.4f}", f"{mi:+.1f}%"],
      ["Std Tox ↓", f"{bs2:.4f}", f"{ps2:.4f}", f"{si:+.1f}%"],
      ["Win Rate ↑", "--", f"{100*wins/NUM_EVAL:.0f}%", "--"]]
if baseline_results.get("rouge_corpus") and ppo_results.get("rouge_corpus"):
    rd = ppo_results["rouge_corpus"]["rougeL"] - baseline_results["rouge_corpus"]["rougeL"]
    td.append(["ROUGE-L ↑", f"{baseline_results['rouge_corpus']['rougeL']:.4f}",
               f"{ppo_results['rouge_corpus']['rougeL']:.4f}", f"{rd:+.4f}"])

t = ax.table(cellText=td, loc="center", cellLoc="center")
t.auto_set_font_size(False); t.set_fontsize(10); t.scale(1.2, 1.5)
for j in range(4):
    t[0,j].set_facecolor("#3498db"); t[0,j].set_text_props(color="white", fontweight="bold")
# Color delta cells green/red based on improvement
for row in range(1, len(td)):
    delta_text = td[row][3]
    if delta_text != "--":
        cell = t[row, 3]
        metric = td[row][0]
        # For ↓ metrics: positive delta = good; for ↑ metrics: positive = good
        is_good = ("↓" in metric and "+" in delta_text) or ("↑" in metric and "+" in delta_text)
        cell.set_facecolor("#d5f5e3" if is_good else "#fadbd8")
ax.set_title("Results", pad=20)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

## Final Report

In [ ]:
pd.DataFrame(log).to_csv(f"{OUTPUT_DIR}/training_log.csv", index=False)

print("="*60)
print("FINAL REPORT")
print("="*60)
print(f"""
Pipeline: FLAN-T5-base + LoRA SFT ({SFT_EPOCHS}ep) -> RM ({len(pref_pairs)} pairs) -> PPO ({PPO_STEPS} steps)

Reward: {REWARD_WEIGHTS}
  Non-toxicity: RoBERTa hate speech classifier
  Faithfulness: DeBERTa NLI
  Quality: ROUGE-L / length heuristic

Toxicity: {bm:.4f} -> {pm:.4f} ({mi:+.1f}%)
Win rate: {wins}/{NUM_EVAL} ({100*wins/NUM_EVAL:.0f}%)
""")
if baseline_results.get("rouge_corpus") and ppo_results.get("rouge_corpus"):
    print(f"ROUGE-L: {baseline_results['rouge_corpus']['rougeL']:.4f} -> {ppo_results['rouge_corpus']['rougeL']:.4f}")

print("\n--- Examples ---")
for i in range(min(3, len(baseline_results["generated_texts"]))):
    print(f"\n[{i+1}] SFT: {baseline_results['generated_texts'][i][:150]}")
    print(f"    PPO: {ppo_results['generated_texts'][i][:150]}")
    print(f"    Tox: {baseline_results['toxicities'][i]:.4f} -> {ppo_results['toxicities'][i]:.4f}")